In [5]:
import pandas as pd
training_df = pd.read_csv(
    "final_training_dataset.csv",
    low_memory=False,         # Processes entire file at once (slightly more memory, fewer warnings)         # Resolve mixed-type warning
)

In [6]:
training_df.head()

,area_name,year,quarter,time_period,rent_growth,population_growth,housing_completions,rent_level,ppi_dublin_all,ppi_nat_ex_dublin_all,ppi_national_all,mortgage_loan_book_n,arrears_90d_n,arrears_720d_n,arrears_90d_rate,arrears_720d_rate
0,Carlow,2018,1,2018Q1,7.174631,1.215278,855.082714,774.45,29.358333,27.108333,29.141667,250.0,35.0,7.0,0.140000,0.028000
1,Carlow,2018,2,2018Q2,5.710050,1.215278,855.082714,774.45,29.283333,29.025000,30.066667,250.0,32.0,7.0,0.128000,0.028000
2,Carlow,2018,3,2018Q3,7.238179,1.215278,855.082714,774.45,29.058333,28.408333,29.708333,252.0,28.0,7.0,0.111111,0.027778
3,Carlow,2018,4,2018Q4,6.610302,1.215278,855.082714,774.45,27.983333,27.691667,28.808333,250.0,25.0,6.0,0.100000,0.024000
4,Carlow,2019,1,2019Q1,6.498452,1.886792,855.082714,826.31,26.150000,26.800000,27.441667,248.0,21.0,4.0,0.084677,0.016129


In [7]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [8]:
glm_df = training_df.copy()
glm_df = glm_df.sort_values(["area_name", "year", "quarter"]).reset_index(drop=True)

print(glm_df.shape)
glm_df.head()

(902, 16)


,area_name,year,quarter,time_period,rent_growth,population_growth,housing_completions,rent_level,ppi_dublin_all,ppi_nat_ex_dublin_all,ppi_national_all,mortgage_loan_book_n,arrears_90d_n,arrears_720d_n,arrears_90d_rate,arrears_720d_rate
0,Carlow,2018,1,2018Q1,7.174631,1.215278,855.082714,774.45,29.358333,27.108333,29.141667,250.0,35.0,7.0,0.140000,0.028000
1,Carlow,2018,2,2018Q2,5.710050,1.215278,855.082714,774.45,29.283333,29.025000,30.066667,250.0,32.0,7.0,0.128000,0.028000
2,Carlow,2018,3,2018Q3,7.238179,1.215278,855.082714,774.45,29.058333,28.408333,29.708333,252.0,28.0,7.0,0.111111,0.027778
3,Carlow,2018,4,2018Q4,6.610302,1.215278,855.082714,774.45,27.983333,27.691667,28.808333,250.0,25.0,6.0,0.100000,0.024000
4,Carlow,2019,1,2019Q1,6.498452,1.886792,855.082714,826.31,26.150000,26.800000,27.441667,248.0,21.0,4.0,0.084677,0.016129


In [9]:
# avoid divide-by-zero
glm_df["mortgage_loan_book_n"] = pd.to_numeric(glm_df["mortgage_loan_book_n"], errors="coerce")
glm_df["arrears_90d_n"] = pd.to_numeric(glm_df["arrears_90d_n"], errors="coerce")
glm_df["arrears_720d_n"] = pd.to_numeric(glm_df["arrears_720d_n"], errors="coerce")
glm_df["rent_level"] = pd.to_numeric(glm_df["rent_level"], errors="coerce")
glm_df["rent_growth"] = pd.to_numeric(glm_df["rent_growth"], errors="coerce")
glm_df["population_growth"] = pd.to_numeric(glm_df["population_growth"], errors="coerce")
glm_df["housing_completions"] = pd.to_numeric(glm_df["housing_completions"], errors="coerce")
glm_df["ppi_national_all"] = pd.to_numeric(glm_df["ppi_national_all"], errors="coerce")
glm_df["ppi_dublin_all"] = pd.to_numeric(glm_df["ppi_dublin_all"], errors="coerce")

# useful transformed features
glm_df["log_housing_completions"] = np.log(glm_df["housing_completions"] + 1)
glm_df["log_rent_level"] = np.log(glm_df["rent_level"].clip(lower=1))
glm_df["log_ppi_national"] = np.log(glm_df["ppi_national_all"].clip(lower=1))
glm_df["log_ppi_dublin"] = np.log(glm_df["ppi_dublin_all"].clip(lower=1))

# supply relative to loan book
glm_df["completions_per_100_loans"] = 100 * (
    glm_df["housing_completions"] / glm_df["mortgage_loan_book_n"].replace(0, np.nan)
)

# optional lagged rent level
glm_df["rent_level_lag_4"] = glm_df.groupby("area_name")["rent_level"].shift(4)
glm_df["rent_level_growth_yoy"] = 100 * (
    (glm_df["rent_level"] - glm_df["rent_level_lag_4"]) /
    glm_df["rent_level_lag_4"].replace(0, np.nan)
)

glm_df.head()

,area_name,year,quarter,time_period,rent_growth,population_growth,housing_completions,rent_level,ppi_dublin_all,ppi_nat_ex_dublin_all,...,arrears_720d_n,arrears_90d_rate,arrears_720d_rate,log_housing_completions,log_rent_level,log_ppi_national,log_ppi_dublin,completions_per_100_loans,rent_level_lag_4,rent_level_growth_yoy
0,Carlow,2018,1,2018Q1,7.174631,1.215278,855.082714,774.45,29.358333,27.108333,...,7.0,0.140000,0.028000,6.752367,6.652153,3.372169,3.379576,342.033085,NaN,NaN
1,Carlow,2018,2,2018Q2,5.710050,1.215278,855.082714,774.45,29.283333,29.025000,...,7.0,0.128000,0.028000,6.752367,6.652153,3.403417,3.377019,342.033085,NaN,NaN
2,Carlow,2018,3,2018Q3,7.238179,1.215278,855.082714,774.45,29.058333,28.408333,...,7.0,0.111111,0.027778,6.752367,6.652153,3.391428,3.369305,339.318537,NaN,NaN
3,Carlow,2018,4,2018Q4,6.610302,1.215278,855.082714,774.45,27.983333,27.691667,...,6.0,0.100000,0.024000,6.752367,6.652153,3.360665,3.331609,342.033085,NaN,NaN
4,Carlow,2019,1,2019Q1,6.498452,1.886792,855.082714,826.31,26.150000,26.800000,...,4.0,0.084677,0.016129,6.752367,6.716970,3.312063,3.263849,344.791417,774.45,6.696365


In [10]:
freq_df = glm_df[[
    "area_name", "year", "quarter",
    "arrears_90d_n", "mortgage_loan_book_n",
    "rent_growth", "population_growth",
    "log_housing_completions", "log_ppi_national"
]].copy()

freq_df = freq_df.dropna()

# keep only valid exposure rows
freq_df = freq_df[freq_df["mortgage_loan_book_n"] > 0].copy()

print(freq_df.shape)
freq_df.head()

(902, 9)


,area_name,year,quarter,arrears_90d_n,mortgage_loan_book_n,rent_growth,population_growth,log_housing_completions,log_ppi_national
0,Carlow,2018,1,35.0,250.0,7.174631,1.215278,6.752367,3.372169
1,Carlow,2018,2,32.0,250.0,5.710050,1.215278,6.752367,3.403417
2,Carlow,2018,3,28.0,252.0,7.238179,1.215278,6.752367,3.391428
3,Carlow,2018,4,25.0,250.0,6.610302,1.215278,6.752367,3.360665
4,Carlow,2019,1,21.0,248.0,6.498452,1.886792,6.752367,3.312063


In [11]:
freq_model = smf.glm(
    formula="""
        arrears_90d_n ~ rent_growth
                       + population_growth
                       + log_housing_completions
                       + log_ppi_national
                       + C(quarter)
    """,
    data=freq_df,
    family=sm.families.Poisson(),
    offset=np.log(freq_df["mortgage_loan_book_n"])
).fit()

print(freq_model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:          arrears_90d_n   No. Observations:                  902
Model:                            GLM   Df Residuals:                      894
Model Family:                 Poisson   Df Model:                            7
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -10576.
Date:                Fri, 10 Apr 2026   Deviance:                       15882.
Time:                        00:22:18   Pearson chi2:                 2.04e+04
No. Iterations:                     5   Pseudo R-squ. (CS):             0.9589
Covariance Type:            nonrobust                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 

In [12]:
freq_df["pred_arrears_90d_n"] = freq_model.predict(
    freq_df,
    offset=np.log(freq_df["mortgage_loan_book_n"])
)

freq_df["pred_arrears_90d_rate"] = (
    freq_df["pred_arrears_90d_n"] / freq_df["mortgage_loan_book_n"]
)

freq_df[[
    "area_name", "year", "quarter",
    "arrears_90d_n", "mortgage_loan_book_n",
    "pred_arrears_90d_n", "pred_arrears_90d_rate"
]].head(10)

,area_name,year,quarter,arrears_90d_n,mortgage_loan_book_n,pred_arrears_90d_n,pred_arrears_90d_rate
0,Carlow,2018,1,35.0,250.0,49.384566,0.197538
1,Carlow,2018,2,32.0,250.0,45.644966,0.182580
2,Carlow,2018,3,28.0,252.0,50.832481,0.201716
3,Carlow,2018,4,25.0,250.0,49.461346,0.197845
4,Carlow,2019,1,21.0,248.0,49.657167,0.200231
5,Carlow,2019,2,25.0,253.0,53.329410,0.210788
6,Carlow,2019,3,29.0,252.0,52.543197,0.208505
7,Carlow,2019,4,30.0,255.0,51.934475,0.203665
8,Carlow,2020,1,29.0,258.0,57.399024,0.222477
9,Carlow,2020,2,29.0,253.0,55.654577,0.219979


In [13]:
sev_df = glm_df[[
    "area_name", "year", "quarter",
    "rent_level",
    "rent_growth", "population_growth",
    "log_housing_completions",
    "log_ppi_national"
]].copy()

sev_df = sev_df.dropna()
sev_df = sev_df[sev_df["rent_level"] > 0].copy()

print(sev_df.shape)
sev_df.head()

(902, 8)


,area_name,year,quarter,rent_level,rent_growth,population_growth,log_housing_completions,log_ppi_national
0,Carlow,2018,1,774.45,7.174631,1.215278,6.752367,3.372169
1,Carlow,2018,2,774.45,5.710050,1.215278,6.752367,3.403417
2,Carlow,2018,3,774.45,7.238179,1.215278,6.752367,3.391428
3,Carlow,2018,4,774.45,6.610302,1.215278,6.752367,3.360665
4,Carlow,2019,1,826.31,6.498452,1.886792,6.752367,3.312063


In [14]:
sev_model = smf.glm(
    formula="""
        rent_level ~ rent_growth
                   + population_growth
                   + log_housing_completions
                   + log_ppi_national
                   + C(quarter)
    """,
    data=sev_df,
    family=sm.families.Gamma(link=sm.families.links.log())
).fit()

print(sev_model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:             rent_level   No. Observations:                  902
Model:                            GLM   Df Residuals:                      894
Model Family:                   Gamma   Df Model:                            7
Link Function:                    log   Scale:                        0.044797
Method:                          IRLS   Log-Likelihood:                -6085.6
Date:                Fri, 10 Apr 2026   Deviance:                       37.945
Time:                        00:22:21   Pearson chi2:                     40.0
No. Iterations:                    10   Pseudo R-squ. (CS):             0.6941
Covariance Type:            nonrobust                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 

/Users/admin/miniconda3/envs/mom_env/lib/python3.11/site-packages/statsmodels/genmod/families/links.py:13: FutureWarning: The log link alias is deprecated. Use Log instead. The log link alias will be removed after the 0.15.0 release.
  warnings.warn(


In [15]:
sev_df["pred_rent_level"] = sev_model.predict(sev_df)

sev_df[[
    "area_name", "year", "quarter",
    "rent_level", "pred_rent_level"
]].head(10)

,area_name,year,quarter,rent_level,pred_rent_level
0,Carlow,2018,1,774.45,1045.845620
1,Carlow,2018,2,774.45,1084.322640
2,Carlow,2018,3,774.45,961.135091
3,Carlow,2018,4,774.45,914.935838
4,Carlow,2019,1,826.31,1022.892463
5,Carlow,2019,2,826.31,941.301077
6,Carlow,2019,3,826.31,923.367634
7,Carlow,2019,4,826.31,882.291010
8,Carlow,2020,1,870.88,940.330970
9,Carlow,2020,2,870.88,916.391208


In [16]:
model_out = glm_df.copy()

model_out = model_out.merge(
    freq_df[[
        "area_name", "year", "quarter",
        "pred_arrears_90d_n", "pred_arrears_90d_rate"
    ]],
    on=["area_name", "year", "quarter"],
    how="left"
)

model_out = model_out.merge(
    sev_df[[
        "area_name", "year", "quarter",
        "pred_rent_level"
    ]],
    on=["area_name", "year", "quarter"],
    how="left"
)

model_out.head()

,area_name,year,quarter,time_period,rent_growth,population_growth,housing_completions,rent_level,ppi_dublin_all,ppi_nat_ex_dublin_all,...,log_housing_completions,log_rent_level,log_ppi_national,log_ppi_dublin,completions_per_100_loans,rent_level_lag_4,rent_level_growth_yoy,pred_arrears_90d_n,pred_arrears_90d_rate,pred_rent_level
0,Carlow,2018,1,2018Q1,7.174631,1.215278,855.082714,774.45,29.358333,27.108333,...,6.752367,6.652153,3.372169,3.379576,342.033085,NaN,NaN,49.384566,0.197538,1045.845620
1,Carlow,2018,2,2018Q2,5.710050,1.215278,855.082714,774.45,29.283333,29.025000,...,6.752367,6.652153,3.403417,3.377019,342.033085,NaN,NaN,45.644966,0.182580,1084.322640
2,Carlow,2018,3,2018Q3,7.238179,1.215278,855.082714,774.45,29.058333,28.408333,...,6.752367,6.652153,3.391428,3.369305,339.318537,NaN,NaN,50.832481,0.201716,961.135091
3,Carlow,2018,4,2018Q4,6.610302,1.215278,855.082714,774.45,27.983333,27.691667,...,6.752367,6.652153,3.360665,3.331609,342.033085,NaN,NaN,49.461346,0.197845,914.935838
4,Carlow,2019,1,2019Q1,6.498452,1.886792,855.082714,826.31,26.150000,26.800000,...,6.752367,6.716970,3.312063,3.263849,344.791417,774.45,6.696365,49.657167,0.200231,1022.892463


In [17]:
model_out["pred_housing_stress"] = (
    model_out["pred_arrears_90d_rate"] * model_out["pred_rent_level"]
)

model_out[[
    "area_name", "year", "quarter",
    "pred_arrears_90d_rate",
    "pred_rent_level",
    "pred_housing_stress"
]].head(10)

,area_name,year,quarter,pred_arrears_90d_rate,pred_rent_level,pred_housing_stress
0,Carlow,2018,1,0.197538,1045.845620,206.594526
1,Carlow,2018,2,0.182580,1084.322640,197.975482
2,Carlow,2018,3,0.201716,961.135091,193.876514
3,Carlow,2018,4,0.197845,914.935838,181.015832
4,Carlow,2019,1,0.200231,1022.892463,204.814283
5,Carlow,2019,2,0.210788,941.301077,198.415142
6,Carlow,2019,3,0.208505,923.367634,192.526538
7,Carlow,2019,4,0.203665,882.291010,179.691453
8,Carlow,2020,1,0.222477,940.330970,209.201859
9,Carlow,2020,2,0.219979,916.391208,201.586423


In [18]:
def minmax_score(s):
    s = pd.to_numeric(s, errors="coerce")
    if s.notna().sum() == 0:
        return pd.Series(np.nan, index=s.index)
    mn, mx = s.min(), s.max()
    if pd.isna(mn) or pd.isna(mx) or mn == mx:
        return pd.Series(50.0, index=s.index)
    return 100 * (s - mn) / (mx - mn)

model_out["pred_housing_stress_score"] = (
    model_out.groupby("time_period")["pred_housing_stress"].transform(minmax_score)
)

In [19]:
cluster_df = model_out[[
    "area_name",
    "pred_arrears_90d_rate",
    "pred_rent_level",
    "rent_growth",
    "population_growth",
    "housing_completions"
]].dropna().copy()

cluster_df = cluster_df.groupby("area_name").mean().reset_index()
cluster_df.head()

,area_name,pred_arrears_90d_rate,pred_rent_level,rent_growth,population_growth,housing_completions
0,Carlow,0.167787,1060.726024,7.677929,1.622367,677.636451
1,Cavan,0.180046,858.353584,8.688649,1.151720,461.817056
2,Clare,0.167661,965.008229,7.626802,1.622322,528.082996
3,Cork,0.169132,1074.585689,5.193550,1.357523,722.989381
4,Donegal,0.173376,869.439598,8.358114,1.396368,450.038190


In [20]:
from sklearn.preprocessing import StandardScaler

features = [
    "pred_arrears_90d_rate",
    "pred_rent_level",
    "rent_growth",
    "population_growth",
    "housing_completions"
]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(cluster_df[features])

In [21]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=4, random_state=42)
cluster_df["cluster"] = kmeans.fit_predict(X_scaled)

cluster_df.sort_values("cluster")

,area_name,pred_arrears_90d_rate,pred_rent_level,rent_growth,population_growth,housing_completions,cluster
0,Carlow,0.167787,1060.726024,7.677929,1.622367,677.636451,0
14,Louth,0.166036,1215.261591,6.413218,1.278505,911.007443,0
2,Clare,0.167661,965.008229,7.626802,1.622322,528.082996,0
3,Cork,0.169132,1074.585689,5.193550,1.357523,722.989381,0
22,Waterford,0.168606,1068.663260,7.770668,1.695006,674.911313,0
6,Galway,0.172826,1003.480695,6.379896,1.648285,634.702808,0
7,Kerry,0.165200,951.688511,8.012354,1.963405,502.524016,0
15,Mayo,0.169638,941.281604,8.589053,1.816605,524.814637,0
9,Kilkenny,0.159333,1074.661928,5.958187,1.523056,623.651298,0
10,Laois,0.168904,1079.175269,7.006913,1.294424,710.123880,0


In [22]:
cluster_summary = cluster_df.groupby("cluster")[features].mean()
cluster_summary

,pred_arrears_90d_rate,pred_rent_level,rent_growth,population_growth,housing_completions
cluster,,,,,
0,0.167386,1047.430635,7.100435,1.589099,655.922553
1,0.175554,874.876893,8.416376,1.363514,464.680677
2,0.163055,1258.893639,4.752637,1.775366,1007.991892
3,0.158099,1461.840853,5.627529,1.436717,1354.059686


In [23]:
cluster_labels = {
    0: "High pressure growth areas",
    1: "Stressed low-supply markets",
    2: "Expanding high-demand markets",
    3: "Mature high-value stable markets"
}

In [24]:

cluster_df["cluster_label"] = cluster_df["cluster"].map(cluster_labels)

In [25]:
model_out = model_out.merge(
    cluster_df[["area_name", "cluster", "cluster_label"]],
    on="area_name",
    how="left"
)

In [26]:
freq_cluster_df = model_out[[
    "arrears_90d_n",
    "mortgage_loan_book_n",
    "rent_growth",
    "population_growth",
    "log_housing_completions",
    "cluster"
]].copy()

freq_cluster_df = freq_cluster_df.dropna()
freq_cluster_df = freq_cluster_df[freq_cluster_df["mortgage_loan_book_n"] > 0].copy()

print(freq_cluster_df.shape)
freq_cluster_df.head()

(1766, 6)


,arrears_90d_n,mortgage_loan_book_n,rent_growth,population_growth,log_housing_completions,cluster
0,35.0,250.0,7.174631,1.215278,6.752367,0
1,32.0,250.0,5.710050,1.215278,6.752367,0
2,28.0,252.0,7.238179,1.215278,6.752367,0
3,25.0,250.0,6.610302,1.215278,6.752367,0
4,21.0,248.0,6.498452,1.886792,6.752367,0


In [27]:
freq_model_clustered = smf.glm(
    formula="""
        arrears_90d_n ~ rent_growth
                       + population_growth
                       + log_housing_completions
                       + C(cluster)
    """,
    data=freq_cluster_df,
    family=sm.families.Poisson(),
    offset=np.log(freq_cluster_df["mortgage_loan_book_n"])
).fit()

print(freq_model_clustered.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:          arrears_90d_n   No. Observations:                 1766
Model:                            GLM   Df Residuals:                     1759
Model Family:                 Poisson   Df Model:                            6
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -27519.
Date:                Fri, 10 Apr 2026   Deviance:                       44175.
Time:                        00:22:29   Pearson chi2:                 8.94e+04
No. Iterations:                     5   Pseudo R-squ. (CS):             0.9245
Covariance Type:            nonrobust                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 

In [28]:
freq_model_clustered_nb = smf.glm(
    formula="""
        arrears_90d_n ~ rent_growth
                       + population_growth
                       + log_housing_completions
                       + C(cluster)
    """,
    data=freq_cluster_df,
    family=sm.families.NegativeBinomial(),
    offset=np.log(freq_cluster_df["mortgage_loan_book_n"])
).fit()

print(freq_model_clustered_nb.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:          arrears_90d_n   No. Observations:                 1766
Model:                            GLM   Df Residuals:                     1759
Model Family:        NegativeBinomial   Df Model:                            6
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -9661.1
Date:                Fri, 10 Apr 2026   Deviance:                       531.25
Time:                        00:22:30   Pearson chi2:                 1.57e+03
No. Iterations:                     8   Pseudo R-squ. (CS):            0.01948
Covariance Type:            nonrobust                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 

/Users/admin/miniconda3/envs/mom_env/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


In [29]:
exp_df = model_out.copy()

exp_df = exp_df.sort_values(["area_name", "year", "quarter"]).reset_index(drop=True)
exp_df.head()

,area_name,year,quarter,time_period,rent_growth,population_growth,housing_completions,rent_level,ppi_dublin_all,ppi_nat_ex_dublin_all,...,completions_per_100_loans,rent_level_lag_4,rent_level_growth_yoy,pred_arrears_90d_n,pred_arrears_90d_rate,pred_rent_level,pred_housing_stress,pred_housing_stress_score,cluster,cluster_label
0,Carlow,2018,1,2018Q1,7.174631,1.215278,855.082714,774.45,29.358333,27.108333,...,342.033085,NaN,NaN,49.384566,0.197538,1045.845620,206.594526,42.820506,0,High pressure growth areas
1,Carlow,2018,2,2018Q2,5.710050,1.215278,855.082714,774.45,29.283333,29.025000,...,342.033085,NaN,NaN,45.644966,0.182580,1084.322640,197.975482,42.549319,0,High pressure growth areas
2,Carlow,2018,3,2018Q3,7.238179,1.215278,855.082714,774.45,29.058333,28.408333,...,339.318537,NaN,NaN,50.832481,0.201716,961.135091,193.876514,43.862510,0,High pressure growth areas
3,Carlow,2018,4,2018Q4,6.610302,1.215278,855.082714,774.45,27.983333,27.691667,...,342.033085,NaN,NaN,49.461346,0.197845,914.935838,181.015832,44.226179,0,High pressure growth areas
4,Carlow,2019,1,2019Q1,6.498452,1.886792,855.082714,826.31,26.150000,26.800000,...,344.791417,774.45,6.696365,49.657167,0.200231,1022.892463,204.814283,41.778278,0,High pressure growth areas


In [30]:
import pandas as pd
import numpy as np

bins = [-np.inf, 25, 45, 65, 80, np.inf]
labels = ["Lower pressure", "Stable", "Watchlist", "High pressure", "Critical"]

if "predicted_classification_glm" not in exp_df.columns:
    exp_df["predicted_classification_glm"] = pd.cut(
        exp_df["pred_housing_stress_score"],
        bins=bins,
        labels=labels
    )

In [31]:
def safe_minmax(s):
    s = pd.to_numeric(s, errors="coerce")
    mn, mx = s.min(), s.max()
    if pd.isna(mn) or pd.isna(mx) or mn == mx:
        return pd.Series(50.0, index=s.index)
    return 100 * (s - mn) / (mx - mn)

# Cross-sectional standardized signals by time_period
exp_df["rent_growth_score_x"] = exp_df.groupby("time_period")["rent_growth"].transform(safe_minmax)
exp_df["population_growth_score_x"] = exp_df.groupby("time_period")["population_growth"].transform(safe_minmax)
exp_df["rent_level_score_x"] = exp_df.groupby("time_period")["rent_level"].transform(safe_minmax)
exp_df["arrears_score_x"] = exp_df.groupby("time_period")["arrears_90d_rate"].transform(safe_minmax)

# Lower supply should increase pressure, so invert completions
exp_df["completions_score_x"] = 100 - exp_df.groupby("time_period")["housing_completions"].transform(safe_minmax)

driver_cols = {
    "rent growth": "rent_growth_score_x",
    "population growth": "population_growth_score_x",
    "rent level": "rent_level_score_x",
    "arrears risk": "arrears_score_x",
    "limited housing supply": "completions_score_x",
}

In [32]:
def top_drivers_for_row(row, driver_map, top_n=3):
    vals = []
    for label, col in driver_map.items():
        v = row.get(col, np.nan)
        if pd.notna(v):
            vals.append((label, float(v)))
    vals = sorted(vals, key=lambda x: x[1], reverse=True)
    return vals[:top_n]

exp_df["top_driver_tuples"] = exp_df.apply(
    lambda r: top_drivers_for_row(r, driver_cols, top_n=3),
    axis=1
)

exp_df["dominant_model_driver"] = exp_df["top_driver_tuples"].apply(
    lambda x: x[0][0] if len(x) > 0 else np.nan
)

exp_df["top_3_model_drivers"] = exp_df["top_driver_tuples"].apply(
    lambda x: ", ".join([t[0] for t in x])
)

exp_df[["area_name", "time_period", "dominant_model_driver", "top_3_model_drivers"]].head()

,area_name,time_period,dominant_model_driver,top_3_model_drivers
0,Carlow,2018Q1,limited housing supply,"limited housing supply, arrears risk, rent level"
1,Carlow,2018Q2,limited housing supply,"limited housing supply, rent level, arrears risk"
2,Carlow,2018Q3,limited housing supply,"limited housing supply, rent growth, rent level"
3,Carlow,2018Q4,limited housing supply,"limited housing supply, rent growth, rent level"
4,Carlow,2019Q1,population growth,"population growth, limited housing supply, ren..."


In [33]:
def factual_explanation(row):
    area = row["area_name"]
    period = row["time_period"]
    cls = row.get("predicted_classification_glm", "Unknown")
    score = row.get("pred_housing_stress_score", np.nan)
    main_driver = row.get("dominant_model_driver", "housing pressure")
    top3 = row.get("top_3_model_drivers", "")
    
    rent_growth = row.get("rent_growth", np.nan)
    pop_growth = row.get("population_growth", np.nan)
    completions = row.get("housing_completions", np.nan)
    arrears = row.get("pred_arrears_90d_rate", row.get("arrears_90d_rate", np.nan))
    rent_level = row.get("pred_rent_level", row.get("rent_level", np.nan))

    return (
        f"{area} ({period}) is predicted to be in {cls} with a housing stress score of "
        f"{score:.1f}. The dominant driver is {main_driver}. "
        f"Key signals are rent growth of {rent_growth:.1f}%, population growth of {pop_growth:.1f}%, "
        f"housing completions of {completions:.0f}, predicted rent level of {rent_level:.0f}, "
        f"and predicted arrears rate of {arrears:.3f}. "
        f"Top drivers: {top3}."
    )

exp_df["factual_explanation"] = exp_df.apply(factual_explanation, axis=1)

In [34]:
def semi_factual_explanation(row):
    area = row["area_name"]
    period = row["time_period"]
    cls = str(row.get("predicted_classification_glm", "Unknown"))
    driver = row.get("dominant_model_driver", "housing pressure")
    score = row.get("pred_housing_stress_score", np.nan)

    if driver == "limited housing supply":
        return (
            f"{area} ({period}) sits close to a lower-pressure scenario. "
            f"If housing completions were moderately higher, the area would likely show some reduction "
            f"in housing stress, even if it remained in {cls}."
        )
    elif driver == "rent growth":
        return (
            f"{area} ({period}) is near a less pressured scenario. "
            f"If rent growth had been somewhat lower this quarter, the predicted housing stress would ease, "
            f"though the area may still remain in {cls}."
        )
    elif driver == "population growth":
        return (
            f"{area} ({period}) is close to a less pressured outcome. "
            f"If demand growth had been slightly weaker or supply had kept pace better, the model would assign "
            f"a somewhat lower stress score, even if classification did not fully change from {cls}."
        )
    elif driver == "arrears risk":
        return (
            f"{area} ({period}) is close to a lower-stress profile. "
            f"If arrears pressure were moderately lower, the county would look more financially stable, "
            f"although it may still remain in {cls}."
        )
    else:
        return (
            f"{area} ({period}) is near a lower-pressure scenario. "
            f"A modest improvement in the main pressure driver would reduce the predicted stress score, "
            f"even if the county remained in {cls}."
        )

exp_df["semi_factual_explanation"] = exp_df.apply(semi_factual_explanation, axis=1)

In [35]:
label_to_upper = {
    "Critical": 80,
    "High pressure": 65,
    "Watchlist": 45,
    "Stable": 25,
    "Lower pressure": 0
}

def next_lower_threshold(current_label):
    current_label = str(current_label)
    if current_label == "Critical":
        return 80, "High pressure"
    if current_label == "High pressure":
        return 65, "Watchlist"
    if current_label == "Watchlist":
        return 45, "Stable"
    if current_label == "Stable":
        return 25, "Lower pressure"
    return None, "Lower pressure"

In [36]:
def counterfactual_explanation(row):
    area = row["area_name"]
    period = row["time_period"]
    cls = str(row.get("predicted_classification_glm", "Unknown"))
    score = row.get("pred_housing_stress_score", np.nan)
    driver = row.get("dominant_model_driver", "housing pressure")

    threshold, target_class = next_lower_threshold(cls)
    if threshold is None:
        return (
            f"{area} ({period}) is already in the lowest pressure band. "
            f"No downward counterfactual is needed."
        )

    gap = score - threshold

    if driver == "limited housing supply":
        return (
            f"To move {area} ({period}) from {cls} to {target_class}, the model suggests improving supply conditions. "
            f"A practical counterfactual is a meaningful increase in completions or a reduction in supply bottlenecks. "
            f"The current stress score of {score:.1f} would need to fall by about {gap:.1f} points."
        )
    elif driver == "rent growth":
        return (
            f"To move {area} ({period}) from {cls} to {target_class}, the model suggests lower rent escalation. "
            f"A plausible counterfactual is slower rent growth over the quarter or stronger supply response. "
            f"The current stress score of {score:.1f} would need to fall by about {gap:.1f} points."
        )
    elif driver == "population growth":
        return (
            f"To move {area} ({period}) from {cls} to {target_class}, the model suggests reducing the demand-supply imbalance. "
            f"A plausible counterfactual is higher supply growth alongside current demand conditions. "
            f"The current stress score of {score:.1f} would need to fall by about {gap:.1f} points."
        )
    elif driver == "arrears risk":
        return (
            f"To move {area} ({period}) from {cls} to {target_class}, the model suggests lower mortgage stress. "
            f"A plausible counterfactual is a reduction in arrears pressure or stronger borrower resilience. "
            f"The current stress score of {score:.1f} would need to fall by about {gap:.1f} points."
        )
    else:
        return (
            f"To move {area} ({period}) from {cls} to {target_class}, the model suggests improving the dominant pressure driver. "
            f"The current stress score of {score:.1f} would need to fall by about {gap:.1f} points."
        )

exp_df["counterfactual_explanation"] = exp_df.apply(counterfactual_explanation, axis=1)

In [37]:
def llm_context_summary(row):
    return (
        f"{row['area_name']} {row['time_period']} | "
        f"class={row.get('predicted_classification_glm', 'NA')} | "
        f"stress_score={row.get('pred_housing_stress_score', np.nan):.1f} | "
        f"pred_arrears_rate={row.get('pred_arrears_90d_rate', np.nan):.3f} | "
        f"pred_rent_level={row.get('pred_rent_level', np.nan):.0f} | "
        f"rent_growth={row.get('rent_growth', np.nan):.1f} | "
        f"population_growth={row.get('population_growth', np.nan):.1f} | "
        f"housing_completions={row.get('housing_completions', np.nan):.0f} | "
        f"cluster={row.get('cluster_label', 'NA')} | "
        f"dominant_driver={row.get('dominant_model_driver', 'NA')}"
    )

exp_df["llm_context_summary"] = exp_df.apply(llm_context_summary, axis=1)

In [38]:
exp_df["topic_tags"] = "housing pressure, rents, arrears, supply, population growth"
exp_df["entity_type"] = "county"
exp_df["source_dataset"] = "signals+rents+ppi+arrears+glm"

In [39]:
export_cols = [
    "area_name",
    "year",
    "quarter",
    "time_period",
    "rent_growth",
    "population_growth",
    "housing_completions",
    "rent_level",
    "ppi_dublin_all",
    "ppi_nat_ex_dublin_all",
    "ppi_national_all",
    "mortgage_loan_book_n",
    "arrears_90d_n",
    "arrears_720d_n",
    "arrears_90d_rate",
    "arrears_720d_rate",
    "pred_arrears_90d_rate",
    "pred_rent_level",
    "pred_housing_stress_score",
    "predicted_classification_glm",
    "cluster",
    "cluster_label",
    "dominant_model_driver",
    "top_3_model_drivers",
    "factual_explanation",
    "semi_factual_explanation",
    "counterfactual_explanation",
    "llm_context_summary",
    "topic_tags",
    "entity_type",
    "source_dataset",
]

llm_export = exp_df[export_cols].copy()
llm_export.head()

,area_name,year,quarter,time_period,rent_growth,population_growth,housing_completions,rent_level,ppi_dublin_all,ppi_nat_ex_dublin_all,...,cluster_label,dominant_model_driver,top_3_model_drivers,factual_explanation,semi_factual_explanation,counterfactual_explanation,llm_context_summary,topic_tags,entity_type,source_dataset
0,Carlow,2018,1,2018Q1,7.174631,1.215278,855.082714,774.45,29.358333,27.108333,...,High pressure growth areas,limited housing supply,"limited housing supply, arrears risk, rent level",Carlow (2018Q1) is predicted to be in Stable w...,Carlow (2018Q1) sits close to a lower-pressure...,To move Carlow (2018Q1) from Stable to Lower p...,Carlow 2018Q1 | class=Stable | stress_score=42...,"housing pressure, rents, arrears, supply, popu...",county,signals+rents+ppi+arrears+glm
1,Carlow,2018,2,2018Q2,5.710050,1.215278,855.082714,774.45,29.283333,29.025000,...,High pressure growth areas,limited housing supply,"limited housing supply, rent level, arrears risk",Carlow (2018Q2) is predicted to be in Stable w...,Carlow (2018Q2) sits close to a lower-pressure...,To move Carlow (2018Q2) from Stable to Lower p...,Carlow 2018Q2 | class=Stable | stress_score=42...,"housing pressure, rents, arrears, supply, popu...",county,signals+rents+ppi+arrears+glm
2,Carlow,2018,3,2018Q3,7.238179,1.215278,855.082714,774.45,29.058333,28.408333,...,High pressure growth areas,limited housing supply,"limited housing supply, rent growth, rent level",Carlow (2018Q3) is predicted to be in Stable w...,Carlow (2018Q3) sits close to a lower-pressure...,To move Carlow (2018Q3) from Stable to Lower p...,Carlow 2018Q3 | class=Stable | stress_score=43...,"housing pressure, rents, arrears, supply, popu...",county,signals+rents+ppi+arrears+glm
3,Carlow,2018,4,2018Q4,6.610302,1.215278,855.082714,774.45,27.983333,27.691667,...,High pressure growth areas,limited housing supply,"limited housing supply, rent growth, rent level",Carlow (2018Q4) is predicted to be in Stable w...,Carlow (2018Q4) sits close to a lower-pressure...,To move Carlow (2018Q4) from Stable to Lower p...,Carlow 2018Q4 | class=Stable | stress_score=44...,"housing pressure, rents, arrears, supply, popu...",county,signals+rents+ppi+arrears+glm
4,Carlow,2019,1,2019Q1,6.498452,1.886792,855.082714,826.31,26.150000,26.800000,...,High pressure growth areas,population growth,"population growth, limited housing supply, ren...",Carlow (2019Q1) is predicted to be in Stable w...,Carlow (2019Q1) is close to a less pressured o...,To move Carlow (2019Q1) from Stable to Lower p...,Carlow 2019Q1 | class=Stable | stress_score=41...,"housing pressure, rents, arrears, supply, popu...",county,signals+rents+ppi+arrears+glm


In [40]:
llm_export.to_csv("housing_explanations_for_agent.csv", index=False)
print("Saved to housing_explanations_for_agent.csv")

Saved to housing_explanations_for_agent.csv


In [41]:
import statsmodels.formula.api as smf
import statsmodels.api as sm
import numpy as np

# Always create a clean modelling dataset
model_data = model_out.copy()

# Drop missing values ONLY for variables used in the model
model_data = model_data.dropna(subset=[
    "arrears_90d_n",
    "rent_growth",
    "population_growth",
    "housing_completions",
    "mortgage_loan_book_n",
    "cluster"
])

# Create log supply variable
model_data["log_housing_completions"] = np.log(
    model_data["housing_completions"].replace(0, np.nan)
)

model_data = model_data.dropna(subset=["log_housing_completions"])

# Fit model
freq_model_clustered = smf.glm(
    formula="""
        arrears_90d_n ~ rent_growth
                       + population_growth
                       + log_housing_completions
                       + C(cluster)
    """,
    data=model_data,
    family=sm.families.Poisson(),
    offset=np.log(model_data["mortgage_loan_book_n"])
).fit()

print(freq_model_clustered.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:          arrears_90d_n   No. Observations:                 1766
Model:                            GLM   Df Residuals:                     1759
Model Family:                 Poisson   Df Model:                            6
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -27520.
Date:                Fri, 10 Apr 2026   Deviance:                       44176.
Time:                        00:22:39   Pearson chi2:                 8.94e+04
No. Iterations:                     5   Pseudo R-squ. (CS):             0.9245
Covariance Type:            nonrobust                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 

In [44]:
coefs = freq_model_clustered.params

beta_rent = coefs.get("rent_growth", 0.0)
beta_pop = coefs.get("population_growth", 0.0)
beta_supply = coefs.get("log_housing_completions", 0.0)

print(beta_rent, beta_pop, beta_supply)

0.005076767269371302 -0.12837426700638327 0.30602459629819134


In [45]:
import numpy as np
import pandas as pd

def target_reduction(row):
    cls = str(row.get("predicted_classification_glm", ""))
    score = row.get("pred_housing_stress_score", np.nan)

    thresholds = {
        "Critical": 80,
        "High pressure": 65,
        "Watchlist": 45,
        "Stable": 25,
    }

    if pd.isna(score) or cls not in thresholds:
        return np.nan

    return max(score - thresholds[cls], 0.0)


def required_arrears_change(row, arrears_weight=0.4):
    reduction_needed = target_reduction(row)
    if pd.isna(reduction_needed):
        return np.nan
    return reduction_needed / arrears_weight


def compute_feature_changes(row):
    current_arrears = row.get("pred_arrears_90d_rate", row.get("arrears_90d_rate", np.nan))
    reduction = required_arrears_change(row)

    if pd.isna(current_arrears) or pd.isna(reduction) or current_arrears <= 0:
        return {}

    # crude mapping from score reduction into target arrears-rate reduction
    target_arrears = max(current_arrears - reduction / 100.0, 1e-6)

    if target_arrears <= 0 or current_arrears <= 0:
        return {}

    log_change = np.log(target_arrears / current_arrears)
    changes = {}

    if beta_rent != 0:
        delta_rent = log_change / beta_rent
        changes["rent_growth_change_pp"] = delta_rent

    if beta_pop != 0:
        delta_pop = log_change / beta_pop
        changes["population_growth_change_pp"] = delta_pop

    if beta_supply != 0:
        delta_supply_log = log_change / beta_supply
        changes["housing_completions_pct_change"] = (np.exp(delta_supply_log) - 1) * 100

    return changes

In [46]:
def precise_counterfactual(row):
    area = row.get("area_name", "This area")
    period = row.get("time_period", "")
    cls = row.get("predicted_classification_glm", "current class")
    score = row.get("pred_housing_stress_score", np.nan)

    changes = compute_feature_changes(row)

    if not changes:
        return (
            f"{area} ({period}) does not have enough model information to compute a precise "
            f"counterfactual explanation."
        )

    parts = []

    if "rent_growth_change_pp" in changes and np.isfinite(changes["rent_growth_change_pp"]):
        val = changes["rent_growth_change_pp"]
        current = row.get("rent_growth", np.nan)
        if pd.notna(current) and current != 0:
            pct = abs(val) / abs(current) * 100
            direction = "reduce" if val < 0 else "increase"
            parts.append(
                f"{direction} rent growth by {abs(val):.2f} percentage points "
                f"(about {pct:.1f}% relative to current rent growth)"
            )

    if "population_growth_change_pp" in changes and np.isfinite(changes["population_growth_change_pp"]):
        val = changes["population_growth_change_pp"]
        current = row.get("population_growth", np.nan)
        if pd.notna(current) and current != 0:
            pct = abs(val) / abs(current) * 100
            direction = "reduce" if val < 0 else "increase"
            parts.append(
                f"{direction} population growth by {abs(val):.2f} percentage points "
                f"(about {pct:.1f}% relative to current population growth)"
            )

    if "housing_completions_pct_change" in changes and np.isfinite(changes["housing_completions_pct_change"]):
        val = changes["housing_completions_pct_change"]
        direction = "increase" if val > 0 else "reduce"
        parts.append(
            f"{direction} housing completions by {abs(val):.1f}%"
        )

    if not parts:
        return (
            f"{area} ({period}) does not have stable enough values to generate a precise "
            f"counterfactual explanation."
        )

    return (
        f"To move {area} ({period}) down from {cls}, the model suggests a reduction of about "
        f"{target_reduction(row):.1f} stress-score points. A model-based counterfactual is to "
        + " OR ".join(parts)
        + "."
    )

In [47]:
exp_df["counterfactual_precise"] = exp_df.apply(precise_counterfactual, axis=1)

In [48]:
exp_df[[
    "area_name",
    "time_period",
    "predicted_classification_glm",
    "pred_housing_stress_score",
    "counterfactual_precise"
]].head(10)

,area_name,time_period,predicted_classification_glm,pred_housing_stress_score,counterfactual_precise
0,Carlow,2018Q1,Stable,42.820506,"To move Carlow (2018Q1) down from Stable, the ..."
1,Carlow,2018Q2,Stable,42.549319,"To move Carlow (2018Q2) down from Stable, the ..."
2,Carlow,2018Q3,Stable,43.862510,"To move Carlow (2018Q3) down from Stable, the ..."
3,Carlow,2018Q4,Stable,44.226179,"To move Carlow (2018Q4) down from Stable, the ..."
4,Carlow,2019Q1,Stable,41.778278,"To move Carlow (2019Q1) down from Stable, the ..."
5,Carlow,2019Q2,Stable,42.558099,"To move Carlow (2019Q2) down from Stable, the ..."
6,Carlow,2019Q3,Stable,42.223535,"To move Carlow (2019Q3) down from Stable, the ..."
7,Carlow,2019Q4,Stable,42.045962,"To move Carlow (2019Q4) down from Stable, the ..."
8,Carlow,2020Q1,Stable,44.722976,"To move Carlow (2020Q1) down from Stable, the ..."
9,Carlow,2020Q2,Stable,44.724010,"To move Carlow (2020Q2) down from Stable, the ..."


In [49]:
exp_df.to_csv("housing_explanations_for_agent.csv", index=False)